### PC演習1：Pythonで期待効用を計算する

次のコードを実行し，演習2の計算結果と一致することを確認せよ．


In [2]:
import math

def expected_value(lottery):
    """
    lottery: list of (probability, outcome)
    """
    return sum(p * x for p, x in lottery)

def expected_utility(lottery, u):
    """
    lottery: list of (probability, outcome)
    u: utility function
    """
    return sum(p * u(x) for p, x in lottery)

u = lambda x: math.sqrt(x)

L = [(0.5, 100), (0.5, 0)]
C = [(1.0, 40)]

EV_L = expected_value(L)
EV_C = expected_value(C)
EU_L = expected_utility(L, u)
EU_C = expected_utility(C, u)

print("E[L] =", EV_L)
print("E[C] =", EV_C)
print("u(L) =", EU_L)
print("u(C) =", EU_C)
print("choose", "L" if EU_L > EU_C else "C")

E[L] = 50.0
E[C] = 40.0
u(L) = 5.0
u(C) = 6.324555320336759
choose C


### PC演習2：効用関数による選択の違い

同じくじ $L=(0.5,100;\ 0.5,0)$ と $C=(1,40)$ を，次の3つの効用関数で比較せよ．

- 危険回避型：$u_1(x)=\sqrt{x}$
- 危険中立型：$u_2(x)=x$
- 危険愛好型：$u_3(x)=x^2$


In [ ]:
u1 = lambda x: math.sqrt(x)
u2 = lambda x: x
u3 = lambda x: x**2

for name, ufun in [("sqrt(x)", u1), ("x", u2), ("x^2", u3)]:
    EU_L = expected_utility(L, ufun)
    EU_C = expected_utility(C, ufun)
    if EU_L > EU_C:
        choice = "L"
    elif EU_L < EU_C:
        choice = "C"
    else:
        choice = "indifferent"

    print(name, "      \t", "u(L) =", EU_L, "u(C) =", EU_C, "choice =", choice)

sqrt(x)       	 u(L) = 5.0 u(C) = 6.324555320336759 choice = C
x       	 u(L) = 50.0 u(C) = 40.0 choice = L
x^2       	 u(L) = 5000.0 u(C) = 1600.0 choice = L


### PC演習3：確実性等価を計算する

$u(x)=\sqrt{x}$ とする．
くじ

$$
L=(0.5,100;\ 0.5,0)
$$

について，次の問いに答えよ．

1. $u(L)$ を計算せよ．
2. 確実性等価 $c$ を求めよ．
3. リスクプレミアムを求めよ．
4. リスクプレミアムの意味を説明せよ．


In [8]:
EU_L = expected_utility(L, u)
certainty_equivalent = EU_L**2
risk_premium = expected_value(L) - certainty_equivalent

print("u(L) =", EU_L)
print("certainty equivalent =", certainty_equivalent)
print("risk premium =", risk_premium)

u(L) = 5.0
certainty equivalent = 25.0
risk premium = 25.0


In [9]:
import math
import random

def crra_utility(x, gamma):
    """
    CRRA utility.
    gamma > 0: risk averse
    gamma = 0: risk neutral
    gamma < 0: risk loving
    """
    x = max(x, 1e-8)
    if abs(gamma - 1.0) < 1e-8:
        return math.log(x)
    return (x ** (1.0 - gamma) - 1.0) / (1.0 - gamma)

def expected_utility_crra(lottery, gamma):
    return sum(p * crra_utility(x, gamma) for p, x in lottery)

def generate_lottery():
    p = random.choice([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
    x_high = random.choice([40, 60, 80, 100, 120, 150])
    x_low = random.choice([0, 5, 10, 20, 30])

    if x_high <= x_low:
        x_high = x_low + 20

    return [(p, x_high), (1 - p, x_low)]

def lottery_to_text(lottery):
    return " + ".join([f"{p:.1f}で{x}" for p, x in lottery])

def generate_question_set(n_questions=8, seed=42):
    random.seed(seed)
    questions = []

    for i in range(n_questions):
        A = generate_lottery()
        B = generate_lottery()
        questions.append({
            "id": i + 1,
            "A": A,
            "B": B,
            "A_text": lottery_to_text(A),
            "B_text": lottery_to_text(B),
        })

    return questions

def show_questions(questions):
    for q in questions:
        print(f"問題 {q['id']}:")
        print(f"  A: {q['A_text']}")
        print(f"  B: {q['B_text']}")

def collect_answers_interactive(questions):
    answers = []
    print("各問題について A または B を入力してください．")

    for q in questions:
        print(f"\n問題 {q['id']}")
        print(f"  A: {q['A_text']}")
        print(f"  B: {q['B_text']}")

        while True:
            ans = input("選択（A/B）: ").strip().upper()
            if ans in ["A", "B"]:
                answers.append(ans)
                break
            print("A または B を入力してください．")

    return answers

def choice_prob_A(EU_A, EU_B, beta=1.0):
    d = beta * (EU_A - EU_B)
    if d >= 0:
        return 1.0 / (1.0 + math.exp(-d))
    ed = math.exp(d)
    return ed / (1.0 + ed)

def negative_log_likelihood(gamma, questions, answers, beta=1.0):
    nll = 0.0
    eps = 1e-12

    for q, ans in zip(questions, answers):
        EU_A = expected_utility_crra(q["A"], gamma)
        EU_B = expected_utility_crra(q["B"], gamma)
        pA = choice_prob_A(EU_A, EU_B, beta=beta)

        if ans == "A":
            nll -= math.log(max(pA, eps))
        else:
            nll -= math.log(max(1 - pA, eps))

    return nll

def estimate_gamma_grid(questions, answers, beta=1.0):
    candidates = [round(-2.0 + 0.05 * i, 2) for i in range(101)]
    best_gamma = None
    best_score = float("inf")

    for gamma in candidates:
        score = negative_log_likelihood(gamma, questions, answers, beta=beta)
        if score < best_score:
            best_gamma = gamma
            best_score = score

    return best_gamma, best_score

def interpret_gamma(gamma, tol=0.15):
    if gamma > tol:
        return "危険回避的"
    if gamma < -tol:
        return "危険愛好的"
    return "危険中立に近い"

questions = generate_question_set(n_questions=8, seed=42)
show_questions(questions)

answers = collect_answers_interactive(questions)
gamma_hat, score = estimate_gamma_grid(questions, answers)

print("推定された gamma =", gamma_hat)
print("解釈：", interpret_gamma(gamma_hat))
print("負の対数尤度 =", round(score, 4))

問題 1:
  A: 0.7で40 + 0.3で0
  B: 0.7で80 + 0.3で5
問題 2:
  A: 0.3で60 + 0.7で0
  B: 0.7で150 + 0.3で30
問題 3:
  A: 0.2で120 + 0.8で20
  B: 0.2で40 + 0.8で0
問題 4:
  A: 0.3で60 + 0.7で30
  B: 0.6で40 + 0.4で30
問題 5:
  A: 0.3で150 + 0.7で30
  B: 0.5で60 + 0.5で20
問題 6:
  A: 0.6で80 + 0.4で0
  B: 0.8で60 + 0.2で20
問題 7:
  A: 0.4で80 + 0.6で5
  B: 0.3で80 + 0.7で0
問題 8:
  A: 0.2で100 + 0.8で0
  B: 0.4で80 + 0.6で30
各問題について A または B を入力してください．

問題 1
  A: 0.7で40 + 0.3で0
  B: 0.7で80 + 0.3で5

問題 2
  A: 0.3で60 + 0.7で0
  B: 0.7で150 + 0.3で30

問題 3
  A: 0.2で120 + 0.8で20
  B: 0.2で40 + 0.8で0

問題 4
  A: 0.3で60 + 0.7で30
  B: 0.6で40 + 0.4で30

問題 5
  A: 0.3で150 + 0.7で30
  B: 0.5で60 + 0.5で20

問題 6
  A: 0.6で80 + 0.4で0
  B: 0.8で60 + 0.2で20

問題 7
  A: 0.4で80 + 0.6で5
  B: 0.3で80 + 0.7で0

問題 8
  A: 0.2で100 + 0.8で0
  B: 0.4で80 + 0.6で30
推定された gamma = -2.0
解釈： 危険愛好的
負の対数尤度 = 0.0
